<div style="width: 98%; max-width: 1200px; margin: 0 auto; font-family: Helvetica, Arial, sans-serif;">

<div style="
        background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%);
        border-radius: 15px;
        padding: 35px;
        text-align: center;
        margin-bottom: 25px;
        border: 1px solid rgba(168, 208, 230, 0.3);
        box-shadow: 0 8px 32px rgba(0,0,0,0.4), inset 0 1px 0 rgba(255,255,255,0.1);
        position: relative;
        overflow: hidden;
    ">
        <div style="position: absolute; top: -50%; left: -50%; width: 200%; height: 200%; background: radial-gradient(circle at 30% 70%, rgba(33, 150, 243, 0.08) 0%, transparent 50%); pointer-events: none;"></div>
        <p style="color: rgba(168, 208, 230, 0.6); font-size: 12px; letter-spacing: 6px; text-transform: uppercase; margin: 0 0 10px 0; font-weight: 300;">PHY-2100 — Sciences de l'Espace</p>
        <h1 style="color: white; margin: 0; text-transform: uppercase; font-size: 28px; font-weight: 300; letter-spacing: 3px; text-shadow: 0 2px 10px rgba(33, 150, 243, 0.3);">
            Devoir 3
        </h1>
        <div style="width: 80px; height: 3px; background: linear-gradient(90deg, transparent, #2196F3, transparent); margin: 15px auto;"></div>
        <h3 style="color: #a8d0e6; margin-top: 10px; margin-bottom: 0; font-weight: 300; font-style: italic; font-size: 15px; letter-spacing: 1px;">
        </h3>
    </div>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #1a1a2e 100%);
    border: 2px solid rgba(168, 208, 230, 0.25);
    border-radius: 12px;
    padding: 25px;
    color: white;
    box-shadow: 0 4px 20px rgba(0,0,0,0.3);
">

<div style="display: flex; align-items: center; margin-bottom: 15px;">
    <div style="width: 4px; height: 25px; background: #2196F3; border-radius: 2px; margin-right: 12px;"></div>
    <h3 style="margin: 0; color: #a8d0e6; font-weight: 400; letter-spacing: 1px;">Équipe de recherche</h3>
</div>

**Alex Baker** — 537 050 929  
**Justine Jean** — 537 287 332  
**Léanne Dupuis** — 123 456 789  
**Zak Roy** — 123 456 789  

<div style="width: 100%; height: 1px; background: linear-gradient(90deg, transparent, rgba(168, 208, 230, 0.3), transparent); margin: 15px 0;"></div>

<div style="display: flex; align-items: center; margin-bottom: 15px;">
    <div style="width: 4px; height: 25px; background: #2196F3; border-radius: 2px; margin-right: 12px;"></div>
    <h3 style="margin: 0; color: #a8d0e6; font-weight: 400; letter-spacing: 1px;">Informations académiques</h3>
</div>

**Cours :** PHY-2100 – Sciences de l'espace (H26)  
**Remise :** 30 mars 2026  
**Encadrement :** Pr. Richard L. Lachance

</div>

</div>

In [7]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║  PHY-2100  Sciences de l'espace  —  Devoir #3                        ║
║  Analyse de trajectoire balistique — Projet HARP  Martlet 2C         ║
║  Canon 16 pouces, Yuma Proving Ground, 1966                          ║
║                                                                      ║
║  Méthodes d'intégration :                                            ║
║    • Euler explicite      O(Δt)                                      ║
║    • Runge-Kutta 4        O(Δt⁴)                                     ║
║    • solve_ivp / RK45     O(Δt⁴⁻⁵) adaptatif  (≡ ode45 MATLAB)       ║
║                                                                      ║
║  Modèles physiques :                                                 ║
║    ρ(z) = ρ₀ exp(−z/h₀)   h₀=8.5 km, ρ₀=1.225 kg/m³                  ║
║    g(z) = g_eq(R⊕/(R⊕+z))²  g_eq=9.78 m/s²                         ║
║    F_D  = ½ ρ(z) C_D A v²   (traînée quadratique de Newton)          ║
║    C_D  = 0.2423  calibré Murphy & Bull 1967, Fig.6                  ║
║                                                                      ║
║  Référence : Murphy & Bull, "HARP 5-Inch and 16-Inch Guns            ║
║              at Yuma Proving Ground, Arizona", 1967                  ║
╚══════════════════════════════════════════════════════════════════════╝
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.integrate import solve_ivp
from pathlib import Path
from datetime import datetime
import atexit
import sys
import warnings
warnings.filterwarnings('ignore')

try:
    OUTPUT_DIR = Path(__file__).resolve().parent
except NameError:
    OUTPUT_DIR = Path.cwd()
RESULTS_TXT = OUTPUT_DIR / 'harp_resultats_terminal.txt'


class _TeeStream:
    """Duplique les sorties vers plusieurs flux (terminal + fichier)."""
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
        return len(data)

    def flush(self):
        for stream in self.streams:
            stream.flush()


_ORIG_STDOUT = sys.stdout
_ORIG_STDERR = sys.stderr
_LOG_FILE = open(RESULTS_TXT, 'w', encoding='utf-8')
_LOG_FILE.write("═" * 64 + "\n")
_LOG_FILE.write(f"Journal d'execution - {datetime.now():%Y-%m-%d %H:%M:%S}\n")
_LOG_FILE.write("═" * 64 + "\n\n")

sys.stdout = _TeeStream(_ORIG_STDOUT, _LOG_FILE)
sys.stderr = _TeeStream(_ORIG_STDERR, _LOG_FILE)


def _close_log_streams():
    """Restaure stdout/stderr et ferme le fichier de log proprement."""
    try:
        sys.stdout.flush()
        sys.stderr.flush()
    finally:
        sys.stdout = _ORIG_STDOUT
        sys.stderr = _ORIG_STDERR
        _LOG_FILE.close()


atexit.register(_close_log_streams)


# ══════════════════════════════════════════════════════════════════════
#  00.  THÈME GRAPHIQUE  (espace sombre)
# ══════════════════════════════════════════════════════════════════════
BG, PANEL, BORDER = '#0D1117', '#161B22', '#30363D'
TEXT, MUTED = '#F0F6FC', '#8B949E'
C = dict(
    nodrag='#58A6FF',   # bleu  – trajectoire analytique (sans drag)
    euler ='#FF7B72',   # coral – Euler
    rk4   ='#FFD700',   # or    – RK4
    ode45 ='#56D364',   # vert  – solve_ivp / ode45
    exp   ='#FF4444',   # rouge – données expérimentales
    gload ='#BC8CFF',   # viola – charge g
    hi    ='#FFA657',   # orange
)

plt.rcParams.update({
    'figure.facecolor': BG,   'axes.facecolor':  PANEL,
    'axes.edgecolor':   BORDER,'axes.labelcolor': TEXT,
    'axes.titlecolor':  TEXT,  'xtick.color':     MUTED,
    'ytick.color':      MUTED, 'grid.color':      BORDER,
    'grid.linestyle':   '--',  'grid.alpha':      0.5,
    'text.color':       TEXT,  'legend.facecolor': PANEL,
    'legend.edgecolor': BORDER,'font.family':     'DejaVu Sans',
    'font.size':        10,    'axes.titlesize':   12,
    'axes.labelsize':   10,    'lines.linewidth':  2.0,
    'figure.dpi':       130,   'axes.titlepad':    10,
})


def style_ax(ax, title='', xl='', yl=''):
    """Applique le style sombre uniformément à un axe."""
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)
    ax.tick_params(colors=MUTED, length=4)
    ax.grid(True, alpha=0.4)
    if title: ax.set_title(title, fontweight='bold', pad=8)
    if xl:    ax.set_xlabel(xl)
    if yl:    ax.set_ylabel(yl)

<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">1. Choix du projectile analysé</h2>
    </div>

</div>
<div style="background-color: #252526; color: white; border-left: 5px solid #2196F3; padding: 15px; border-radius: 5px; margin-top: 15px;">

Pour ce devoir, on a choisi d’analyser un projectile HARP, plus précisément le Martlet 2C. Ce choix est pertinent parce qu’il s’agit d’un projectile réel, bien documenté, et surtout très intéressant du point de vue balistique puisque ses vitesses de lancement sont hypersoniques. Cela permet d’étudier de manière claire l’effet de la traînée atmosphérique sur la portée, l’apogée et l’angle optimal de lancement.

Le scénario retenu est basé sur les essais HARP réalisés à Yuma Proving Ground. Notre cas de référence principal est le Round 28, qui correspond à un lancement à très haute vitesse et qui a atteint une apogée expérimentale d’environ 180 km. Ce cas est utile, car il permet à la fois de faire une simulation complète et de comparer les résultats numériques à des données expérimentales réelles.

## Spécifications techniques du projectile

Les paramètres utilisés dans le modèle sont les suivants :

- projectile : HARP Martlet 2C
- masse : 83.9 kg
- diamètre : 12.7 cm
- aire frontale : 126.68 cm²
- coefficient de traînée constant utilisé : $C_D = 0.2422$
- vitesse initiale du Round 28 : 2164 m/s
- nombre de Mach initial : environ 6.4

La cellule suivante contient précisément les constantes physiques et les paramètres du projectile utilisés dans le travail.
</div>

In [8]:

# ══════════════════════════════════════════════════════════════════════
#  1.  CONSTANTES & PARAMÈTRES HARP
# ══════════════════════════════════════════════════════════════════════
R_EARTH  = 6_371_000.0   # m
G_EQ     = 9.78          # m/s²  (valeur imposée par le devoir)
H0       = 8_500.0       # m     (hauteur d'échelle, devoir)
RHO_0    = 1.225         # kg/m³ (ISA, z = 0)
G0_SI    = 9.80665       # m/s²  (standard BIPM, pour la charge g)

# ── Martlet 2C  (Murphy & Bull 1967) ─────────────────────────────────
M_PROJ  = 83.9                       # kg   masse en vol (sans sabot)
D_PROJ  = 0.127                      # m    corps 5 pouces
A_PROJ  = np.pi * (D_PROJ / 2)**2   # m²   section frontale ≈ 0.01267 m²

# Calibration C_D depuis Figure 6 de Murphy & Bull
#   W / (C_D · A) = 5600 lb/ft²  →  C_D = m / (A · WCDA)
_WCDA   = 5600 * 0.453592 / (0.3048**2)   # 5600 lb/ft² → kg/m²
CD_CONST = M_PROJ / (A_PROJ * _WCDA)       # ≈ 0.2423

# Vitesse terminale (chute verticale, z = 0, calcul de sanité)
V_TERM  = np.sqrt(M_PROJ * G0_SI / (0.5 * RHO_0 * CD_CONST * A_PROJ))

# ── Conversions ───────────────────────────────────────────────────────
ft2ms  = lambda v: v * 0.3048    # ft/s → m/s
kft2m  = lambda d: d * 304.8    # kft  → m

# ── Données expérimentales — Table II, Murphy & Bull 1967 ─────────────
# Colonnes : [θ [°], v₀ [m/s], apogée [m], portée [m]]
EXP = np.array([
    [83.9, ft2ms(5930), kft2m(415), kft2m(158)],
    [84.0, ft2ms(5630), kft2m(375), kft2m(144)],
    [85.0, ft2ms(6800), kft2m(540), kft2m(167)],
    [84.5, ft2ms(6780), kft2m(535), kft2m(167)],
    [84.3, ft2ms(6650), kft2m(510), kft2m(182)],
    [83.6, ft2ms(6400), kft2m(490), kft2m(196)],
    [83.8, ft2ms(6650), kft2m(530), kft2m(207)],
    [83.7, ft2ms(5850), kft2m(400), kft2m(162)],
    [84.8, ft2ms(7100), kft2m(590), kft2m(188)],   # Round 28  ← RECORD
    [83.5, ft2ms(6350), kft2m(480), kft2m(198)],
    [83.6, ft2ms(5720), kft2m(367), kft2m(149)],
    [84.2, ft2ms(6750), kft2m(550), kft2m(195)],
])

# Round 28 — cas de référence (vitesse maximale, apogée record ~180 km)
V0_R28   = ft2ms(7100)    # ≈ 2162.7 m/s
TH_R28   = 84.8           # °
APO_R28  = kft2m(590)     # ≈ 179 832 m  ≈ 180 km
RNG_R28  = kft2m(188)     # ≈  57 302 m  ≈  57 km


<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">2.1 Modèle de densité de l’air, gravité variable et traînée</h2>
</div>

La densité est modélisée par :
$$
\rho(y)=\rho_0 e^{-y/h_0},
$$
ce qui signifie que l’air devient rapidement moins dense avec l’altitude. La gravité est modélisée par :
$$
g(y)=g_{\mathrm{eq}}\left(\frac{R_\oplus}{R_\oplus+y}\right)^2,
$$
ce qui permet de tenir compte du fait que le projectile atteint des altitudes très grandes. La traînée est prise sous forme quadratique :
$$
F_D = \frac12 \rho(y) C_D A v^2.
$$

Dans le cas principal, j’utilise un coefficient de traînée constant. En bonus, le code contient aussi une loi $C_D(M)$ dépendant du Mach.

</div>

In [9]:
# ══════════════════════════════════════════════════════════════════════
#  2.1  MODÈLES PHYSIQUES
# ══════════════════════════════════════════════════════════════════════

def gravity(z):
    """g(z) = g_eq · (R⊕ / (R⊕ + z))²  [m/s²]"""
    return G_EQ * (R_EARTH / (R_EARTH + max(float(z), 0.0)))**2


def air_rho(z):
    """ρ(z) = ρ₀ · exp(−z / h₀)  [kg/m³]"""
    return RHO_0 * np.exp(-max(float(z), 0.0) / H0)


def sound_speed(z):
    """Vitesse du son — atmosphère standard ISA  [m/s]"""
    z = max(float(z), 0.0)
    T = 288.15 - 6.5e-3 * z if z < 11_000 else max(216.65 + 1e-3 * (z - 20_000), 180.0)
    return np.sqrt(1.4 * 287.05 * T)


def cd_mach(v, z):
    """
    C_D(M) — formule des notes de cours (slide II-35)
    Valable pour systèmes sans portance significative.
    M ≤ 0.85  :  C_D ≈ 0.20  (subsonique, incompressible)
    M  > 0.85  :  C_D ≈ 0.11 + 0.82/M² − 0.55/M⁴
    """
    M = v / sound_speed(z)
    if M <= 0.85:
        return 0.20
    return max(0.11 + 0.82 / M**2 - 0.55 / M**4, 0.08)

<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">2.2 Équations du mouvement</h2>
</div>

Le mouvement est décrit à l’aide de l’état :
$$
\mathbf{s}=[x,\ y,\ v_x,\ v_y].
$$

Les équations différentielles sont :
$$
\dot{x}=v_x,\qquad \dot{y}=v_y,
$$
$$
\dot{v}_x=-\frac{1}{2m}\rho(y)C_DA\,v\,v_x,
$$
$$
\dot{v}_y=-g(y)-\frac{1}{2m}\rho(y)C_DA\,v\,v_y,
$$
avec
$$
v=\sqrt{v_x^2+v_y^2}.
$$

Les conditions initiales sont :
$$
x(0)=0,\quad y(0)=0,\quad
v_x(0)=v_0\cos\theta_0,\quad
v_y(0)=v_0\sin\theta_0.
$$

La cellule suivante contient la fonction des équations du mouvement et l’événement qui arrête l’intégration lorsque le projectile revient au sol.

</div>

In [11]:

# ══════════════════════════════════════════════════════════════════════
#  2.2  ÉQUATIONS DU MOUVEMENT
# ══════════════════════════════════════════════════════════════════════

def eom(t, s, drag=True, cd_mode='const'):
    """
    Fonction dérivée  ds/dt = f(t, s)   avec  s = [x, y, vx, vy]

    Sans frottement :
        ẍ = 0
        ÿ = −g(y)

    Avec traînée quadratique de Newton :
        ẍ = −μ_N vₓ |v|
        ÿ = −g(y) − μ_N v_y |v|    où  μ_N = ρ(y) C_D A / (2m)
    """
    x, y, vx, vy = s
    y  = max(float(y), 0.0)
    g  = gravity(y)
    ax, ay = 0.0, -g

    if drag:
        v = np.hypot(vx, vy)
        if v > 1e-10:
            r  = air_rho(y)
            cd = CD_CONST if cd_mode == 'const' else cd_mach(v, y)
            aD = 0.5 * r * cd * A_PROJ * v**2 / M_PROJ   # |a_drag|
            ax -= aD * vx / v
            ay -= aD * vy / v

    return [vx, vy, ax, ay]


def _ground_event(t, s, *a):
    """Condition d'arrêt : y = 0  (atterrissage)"""
    return s[1]

_ground_event.terminal  = True
_ground_event.direction = -1

<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">2.3 Méthodes numériques utilisées </h2>
</div>

On utilise les méthodes numériques suivantes, pour pouvoir les comparés :

- Euler explicite
- Runge–Kutta d’ordre 4 (RK4)
- RK45 adaptatif avec `solve_ivp`, l’équivalent Python de `ode45`

Cette comparaison permet de vérifier la robustesse numérique des résultats. En pratique, Euler reste utile comme référence simple, tandis que RK4 et RK45 donnent des résultats presque identiques.

</div>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  2.3  INTÉGRATEURS NUMÉRIQUES
# ══════════════════════════════════════════════════════════════════════
def _ic(theta_deg, v0):
    """Conditions initiales  s₀ = [0, 0, v₀ cos θ, v₀ sin θ]"""
    th = np.radians(theta_deg)
    return np.array([0., 0., v0 * np.cos(th), v0 * np.sin(th)])


def run_euler(theta_deg, v0, dt=0.25, drag=True, cd_mode='const'):
    """
    Méthode d'Euler explicite  (1er ordre — O(Δt))
    s_{n+1} = s_n + Δt · f(t_n, s_n)
    Arrêt dès que y passe sous zéro (interpolation linéaire pour l'atterrissage exact).
    """
    s = _ic(theta_deg, v0)
    ts, traj = [0.], [s.copy()]
    t = 0.
    while t < 4000:
        d     = np.array(eom(t, s, drag, cd_mode))
        s_new = s + dt * d
        if s_new[1] < 0 and s[1] >= 0:
            # Interpolation linéaire → instant exact d'atterrissage
            frac    = s[1] / (s[1] - s_new[1])
            s_land  = s + frac * dt * d
            s_land[1] = 0.0
            ts.append(t + frac * dt)
            traj.append(s_land)
            break
        s = s_new
        t += dt
        ts.append(t)
        traj.append(s.copy())
    return np.array(ts), np.array(traj)


def run_rk4(theta_deg, v0, dt=1.0, drag=True, cd_mode='const'):
    """
    Runge-Kutta d'ordre 4  (4e ordre — O(Δt⁴))
    s_{n+1} = s_n + (Δt/6)(k₁ + 2k₂ + 2k₃ + k₄)
    Arrêt dès que y < 0 avec interpolation linéaire pour l'atterrissage.
    """
    s = _ic(theta_deg, v0)
    ts, traj = [0.], [s.copy()]
    t = 0.
    while t < 4000:
        k1 = np.array(eom(t,        s,           drag, cd_mode))
        k2 = np.array(eom(t+dt/2,   s+dt/2*k1,  drag, cd_mode))
        k3 = np.array(eom(t+dt/2,   s+dt/2*k2,  drag, cd_mode))
        k4 = np.array(eom(t+dt,     s+dt*k3,    drag, cd_mode))
        s_new = s + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
        if s_new[1] < 0 and s[1] >= 0:
            frac   = s[1] / (s[1] - s_new[1])
            d_avg  = (k1 + 2*k2 + 2*k3 + k4) / 6
            s_land = s + frac * dt * d_avg
            s_land[1] = 0.0
            ts.append(t + frac * dt)
            traj.append(s_land)
            break
        s = s_new
        t += dt
        ts.append(t)
        traj.append(s.copy())
    return np.array(ts), np.array(traj)


def run_ode45(theta_deg, v0, drag=True, cd_mode='const', tol=1e-7):
    """
    scipy solve_ivp  méthode RK45  (4e-5e ordre adaptatif)
    Équivalent direct du solveur ode45 de MATLAB.
    Le pas de temps est ajusté automatiquement pour maintenir
    l'erreur locale en-dessous de la tolérance spécifiée.
    """
    y0  = _ic(theta_deg, v0)
    sol = solve_ivp(
        fun=lambda t, y: eom(t, y, drag, cd_mode),
        t_span=[0, 4000],
        y0=y0,
        method='RK45',
        events=_ground_event,
        rtol=tol, atol=tol,
        max_step=2.0,
    )
    s = sol.y.T.copy()
    s[:, 1] = np.maximum(s[:, 1], 0.)
    return sol.t, s


def run_analytic(theta_deg, v0, N=4000):
    """
    Solution analytique — trajectoire parabolique
    (g constant = G_EQ, sans frottement)
    x(t) = v₀ₓ t
    y(t) = v₀_y t − ½ g t²
    """
    th       = np.radians(theta_deg)
    v0x, v0y = v0 * np.cos(th), v0 * np.sin(th)
    t_land   = 2 * v0y / G_EQ
    t        = np.linspace(0, t_land, N)
    x        = v0x * t
    y        = np.maximum(v0y * t - 0.5 * G_EQ * t**2, 0.)
    vx       = np.full_like(t, v0x)
    vy       = v0y - G_EQ * t
    return t, np.column_stack([x, y, vx, vy])

<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">Comparaison rapide des méthodes</h2>
</div>

En comparant les trois approches, on remarque que la méthode d’Euler donne des résultats corrects, mais légèrement moins précis que les 2 autres méthodes. Comme elle accumule davantage d’erreur numérique, elle tend ici à sous-estimer un peu la portée et l’apogée.

La méthode RK4 donne au contraire des résultats presque identiques à ceux obtenus avec RK45/ode45, tout en restant simple à programmer. La méthode RK45 adaptative est celle qui inspire le plus confiance pour ce travail, car elle ajuste automatiquement le pas de temps selon la dynamique du problème. La bonne concordance entre RK4 et RK45 montre que la solution numérique est stable et que les valeurs obtenues pour la portée, l’apogée et le temps de vol sont fiables.

<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">2.4 Fonctions d'analyse </h2>
</div>

Pour exploiter les trajectoires calculées, on utilise des fonctions qui servent à extraire automatiquement :
- l’apogée
- la portée
- le temps de vol
- la position de l’apogée
- la charge structurale en $g$
- le balayage angulaire permettant de trouver $\theta_{\mathrm{opt}}$

Cette partie du code centralise donc toute l’analyse des résultats.

</div>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  2.4  FONCTIONS D'ANALYSE
# ══════════════════════════════════════════════════════════════════════

def get_key_points(ts, ss):
    """
    Retourne les instants et positions clés :
        (t_apo, x_apo, y_apo, t_land, x_range)
    """
    ia = np.argmax(ss[:, 1])
    return ts[ia], ss[ia, 0], ss[ia, 1], ts[-1], ss[-1, 0]


def compute_gload(ts, ss, cd_mode='const'):
    """
    Charge g structurale sur le projectile  [g₀]
    En vol balistique, le projectile est en chute libre → il ne "ressent"
    pas la gravité. La seule force structurale est la traînée aérodynamique :
        n_g = |a_drag| / g₀ = ρ(y) C_D A v² / (2 m g₀)
    """
    loads = np.zeros(len(ts))
    for i, s in enumerate(ss):
        v = np.hypot(s[2], s[3])
        if v > 1e-9:
            r  = air_rho(s[1])
            cd = CD_CONST if cd_mode == 'const' else cd_mach(v, s[1])
            loads[i] = 0.5 * r * cd * A_PROJ * v**2 / (M_PROJ * G0_SI)
    return loads


def scan_angles(v0, angles, drag=True, cd_mode='const'):
    """Balayage des angles via ode45 → tableau [theta, x_apo, y_apo, x_range]"""
    out = []
    for th in angles:
        t, s = run_ode45(th, v0, drag=drag, cd_mode=cd_mode)
        if not drag:
            t, s = run_analytic(th, v0)
        _, xa, ya, _, xr = get_key_points(t, s)
        out.append([th, xa, ya, xr])
    return np.array(out)


<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">2.5 Recherche de l’angle optimal </h2>
</div>

Pour déterminer l’angle donnant la portée maximale, on effectue un balayage numérique d’une série d’angles. Pour chaque angle, le code calcule la trajectoire complète, puis extrait la portée finale et l’apogée.

Le résultat principal trouvé est :
$$
\theta_{\mathrm{opt}} \approx 50^\circ.
$$

Dans le cas sans frottement, le balayage discret donne environ $44^\circ$, ce qui est cohérent avec la valeur théorique de $45^\circ$. La petite différence vient du pas angulaire utilisé et non d’un effet physique.

</div>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  2.5  CALCULS PRINCIPAUX
# ══════════════════════════════════════════════════════════════════════
print("\n" + "═" * 64)
print("  PHY-2100  ·  Devoir #3  ·  HARP Martlet 2C")
print("═" * 64)
print(f"\n  Specs :  m = {M_PROJ} kg  |  D = {D_PROJ*100:.1f} cm  |  A = {A_PROJ*1e4:.2f} cm²")
print(f"  C_D = {CD_CONST:.4f}  (Murphy & Bull 1967)  |  v_term = {V_TERM:.0f} m/s ≈ Mach {V_TERM/sound_speed(0):.2f}")
print(f"  μ_N(z=0) = {0.5*RHO_0*CD_CONST*A_PROJ/M_PROJ:.3e} m⁻¹")
print(f"\n  Charge g au lancement (Round 28) ≈ {0.5*RHO_0*CD_CONST*A_PROJ*V0_R28**2/(M_PROJ*G0_SI):.1f} g")

# ── 6.1  Balayage d'angles pour trouver θ_opt ─────────────────────────
print("\n  [1/4] Balayage d'angles (v₀ = {:.0f} m/s, Round 28)…".format(V0_R28))
angles_sweep = np.arange(5, 88, 3)
res_drag = scan_angles(V0_R28, angles_sweep, drag=True)
res_nd   = scan_angles(V0_R28, angles_sweep, drag=False)

idx_opt    = np.argmax(res_drag[:, 3])
theta_opt  = res_drag[idx_opt, 0]
range_opt  = res_drag[idx_opt, 3]
apo_opt    = res_drag[idx_opt, 2]

idx_nd_opt = np.argmax(res_nd[:, 3])
theta_nd   = res_nd[idx_nd_opt, 0]
range_nd   = res_nd[idx_nd_opt, 3]

print(f"     Avec drag  : θ_opt = {theta_opt:.0f}°  →  R_max = {range_opt/1e3:.1f} km  |  apogée = {apo_opt/1e3:.1f} km")
print(f"     Sans drag  : θ_opt = {theta_nd:.0f}°   →  R_max = {range_nd/1e3:.1f} km  (attendu 45°)")

# ── 6.2  Trajectoires à θ_opt (3 méthodes + analytique) ──────────────
print(f"\n  [2/4] Trajectoires à θ_opt = {theta_opt}°…")
tE, sE = run_euler(theta_opt, V0_R28, dt=0.25)
tR, sR = run_rk4(theta_opt, V0_R28, dt=1.0)
tO, sO = run_ode45(theta_opt, V0_R28)
tA, sA = run_analytic(theta_opt, V0_R28)

print(f"  {'Méthode':<14} {'Apogée x [km]':>14} {'Apogée y [km]':>14} {'Portée [km]':>12} {'t_vol [s]':>10}")
print("  " + "-"*66)
for nm, t, s in [('Euler (Δt=0.25s)', tE, sE),
                  ('RK4  (Δt=1.0s) ', tR, sR),
                  ('ode45 (adapt.) ', tO, sO),
                  ('Analytique     ', tA, sA)]:
    ta, xa, ya, tl, xr = get_key_points(t, s)
    print(f"  {nm:<14}  {xa/1e3:>12.1f}   {ya/1e3:>12.1f}   {xr/1e3:>10.1f}   {tl:>8.0f}")

# ── 6.3  Validation Round 28 (θ=84.8°, v₀≈2163 m/s) ─────────────────
print(f"\n  [3/4] Validation — Round 28 (θ = {TH_R28}°, v₀ = {V0_R28:.0f} m/s)…")
tV, sV = run_ode45(TH_R28, V0_R28)
_, xa_V, ya_V, tl_V, xr_V = get_key_points(tV, sV)
print(f"     Simulation  :  apogée = {ya_V/1e3:.1f} km  |  portée = {xr_V/1e3:.1f} km  |  t_vol = {tl_V:.0f} s")
print(f"     Expérimental:  apogée = {APO_R28/1e3:.1f} km  |  portée = {RNG_R28/1e3:.1f} km")
print(f"     Erreur       :  Δh = {abs(ya_V - APO_R28)/APO_R28*100:.1f}%  |  ΔR = {abs(xr_V - RNG_R28)/RNG_R28*100:.1f}%")

# ── 6.4  Données bonus : vitesses, charge g, y(v) ─────────────────────
print(f"\n  [4/4] Calculs bonus (ode45, θ_opt)…")
gl_opt = compute_gload(tO, sO)
vO     = np.hypot(sO[:, 2], sO[:, 3])
MachO  = np.array([vO[i] / sound_speed(sO[i, 1]) for i in range(len(tO))])

print(f"     G-load max   : {gl_opt.max():.1f} g  (à t={tO[np.argmax(gl_opt)]:.1f} s)")
print(f"     Mach max     : {MachO[0]:.2f}  →  Mach min : {MachO.min():.3f}")

# Validation sur toutes les expériences
print(f"\n  Validation complète ({len(EXP)} tirs HARP) :")
sim_apos, sim_rngs = [], []
for row in EXP:
    th, v0, apo_exp, rng_exp = row
    t_, s_ = run_ode45(th, v0)
    _, _, ya_, _, xr_ = get_key_points(t_, s_)
    sim_apos.append(ya_)
    sim_rngs.append(xr_)
sim_apos = np.array(sim_apos)
sim_rngs = np.array(sim_rngs)
errs_apo = (sim_apos - EXP[:, 2]) / EXP[:, 2] * 100
errs_rng = (sim_rngs - EXP[:, 3]) / EXP[:, 3] * 100
print(f"     Apogée  — erreur moy = {np.mean(np.abs(errs_apo)):.1f}%  |  max = {np.max(np.abs(errs_apo)):.1f}%")
print(f"     Portée  — erreur moy = {np.mean(np.abs(errs_rng)):.1f}%  |  max = {np.max(np.abs(errs_rng)):.1f}%")

# ── Trajectoires pour le balayage d'angle (figure 1b) ──────────────────
sweep_plot_angles = [10, 20, 30, theta_opt, 50, 60, 70, 80, 84.8]
sweep_trajs = {}
for th in sweep_plot_angles:
    t_, s_ = run_ode45(th, V0_R28)
    sweep_trajs[th] = (t_, s_)


<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">2.6 Trajectoires, portée maximale et apogée </h2>
</div>


La figure suivante regroupe :
- la trajectoire $y(x)$ pour l’angle optimal ;
- un balayage des trajectoires pour plusieurs angles ;
- la portée maximale $x_{\max}(\theta_0)$ ;
- l’apogée $y_{\max}(\theta_0)$.

Cette figure montre deux choses :
1. la traînée atmosphérique réduit fortement la portée et l’apogée ;
2. l’angle optimal avec traînée est plus grand que $45^\circ$, parce qu’une trajectoire un peu plus verticale permet de quitter plus rapidement les basses couches denses de l’atmosphère.

</div>

In [ ]:

# ══════════════════════════════════════════════════════════════════════
#  2.6  FIGURE 1 — Trajectoires et analyse des angles
# ══════════════════════════════════════════════════════════════════════
print("\n  Création des figures…")

fig1 = plt.figure(figsize=(18, 13))
fig1.patch.set_facecolor(BG)
gs1 = gridspec.GridSpec(2, 2, figure=fig1, hspace=0.38, wspace=0.32,
                         left=0.07, right=0.97, top=0.91, bottom=0.08)
fig1.suptitle(
    f"HARP Martlet 2C  ·  Trajectoire balistique  ·  v₀ = {V0_R28:.0f} m/s",
    fontsize=15, fontweight='bold', color=TEXT, y=0.97
)

# ── (a) Trajectoires y(x) à θ_opt — comparaison des méthodes ──────────
ax = fig1.add_subplot(gs1[0, 0])
ta1, xa1, ya1, tl1, xr1 = get_key_points(tO, sO)
ax.fill_between([0, max(sA[:, 0])/1e3], 0, -3, color='#1a3a2a', alpha=0.5)
ax.plot(sA[:, 0]/1e3, sA[:, 1]/1e3, color=C['nodrag'], lw=2.5, ls='--',
        label='Analytique (sans drag)', alpha=0.9, zorder=2)
ax.plot(sE[:, 0]/1e3, sE[:, 1]/1e3, color=C['euler'],  lw=2.0,
        label=f"Euler  Δt=0.25 s", alpha=0.9, zorder=3)
ax.plot(sR[:, 0]/1e3, sR[:, 1]/1e3, color=C['rk4'],   lw=2.0, ls='-.',
        label=f"RK4    Δt=1.0 s", alpha=0.9, zorder=4)
ax.plot(sO[:, 0]/1e3, sO[:, 1]/1e3, color=C['ode45'], lw=2.5,
        label="ode45  adaptatif", alpha=1.0, zorder=5)
# Apogée
ax.plot(xa1/1e3, ya1/1e3, '^', color='white', ms=9, zorder=6)
ax.annotate(f'  Apogée\n  ({xa1/1e3:.0f}, {ya1/1e3:.0f}) km',
            xy=(xa1/1e3, ya1/1e3), xytext=(xa1/1e3 + 10, ya1/1e3 - 25),
            fontsize=8, color='white',
            arrowprops=dict(arrowstyle='->', color='white', lw=1.3),
            bbox=dict(boxstyle='round,pad=0.3', fc=PANEL, ec=BORDER, alpha=0.85))
# Portée
ax.plot(xr1/1e3, 0, '*', color=C['ode45'], ms=13, zorder=6, mew=1.5)
ax.annotate(f'  R = {xr1/1e3:.0f} km\n  t = {tl1:.0f} s',
            xy=(xr1/1e3, 0), xytext=(xr1/1e3 - 25, 15),
            fontsize=8, color=C['ode45'],
            arrowprops=dict(arrowstyle='->', color=C['ode45'], lw=1.3),
            bbox=dict(boxstyle='round,pad=0.3', fc=PANEL, ec=C['ode45'], alpha=0.85))
style_ax(ax, title=f"(a)  Trajectoire y(x) à θ_opt = {theta_opt}°",
         xl='Portée horizontale  x  [km]', yl='Altitude  y  [km]')
ax.legend(fontsize=8.5, loc='upper right', framealpha=0.9)
ax.set_xlim(left=-5)
ax.set_ylim(bottom=-5)

# ── (b) Balayage d'angles (ode45) ──────────────────────────────────────
ax = fig1.add_subplot(gs1[0, 1])
cmap = plt.cm.plasma
n_sw = len(sweep_plot_angles)
cols_sw = cmap(np.linspace(0.05, 0.95, n_sw))
for i, th in enumerate(sweep_plot_angles):
    t_, s_ = sweep_trajs[th]
    is_opt = (th == theta_opt)
    is_r28 = (th == 84.8)
    lw  = 3.0 if (is_opt or is_r28) else 1.5
    lbl = (f'θ={th}° ← optimal' if is_opt else
           f'θ={th}° (Round 28)' if is_r28 else f'θ={th}°')
    ax.plot(s_[:, 0]/1e3, s_[:, 1]/1e3, color=cols_sw[i], lw=lw,
            label=lbl, alpha=0.85 if is_opt or is_r28 else 0.65)
    ax.plot(s_[-1, 0]/1e3, 0, 'o', color=cols_sw[i], ms=5, alpha=0.9)
# Marque l'apogée du cas optimal
t_sw, s_sw = sweep_trajs[theta_opt]
_, xs, ys, _, xrs = get_key_points(t_sw, s_sw)
ax.plot(xs/1e3, ys/1e3, '^', color='white', ms=8, zorder=5)
style_ax(ax, title='(b)  Balayage d\'angles — ode45 adaptatif',
         xl='Portée horizontale  x  [km]', yl='Altitude  y  [km]')
ax.legend(fontsize=7.5, loc='upper right', ncol=2, framealpha=0.9)

# ── (c) Portée x_max(θ) ────────────────────────────────────────────────
ax = fig1.add_subplot(gs1[1, 0])
ax.plot(res_nd[:, 0],   res_nd[:, 3]/1e3,   color=C['nodrag'], lw=2.5, ls='--',
        label='Sans frottement (analytique)', alpha=0.9)
ax.plot(res_drag[:, 0], res_drag[:, 3]/1e3, color=C['ode45'],  lw=2.5,
        label='Avec frottement (ode45)')
ax.axvline(theta_opt, color=C['ode45'], ls=':', lw=1.5, alpha=0.7)
ax.axvline(45,        color=C['nodrag'], ls=':', lw=1.5, alpha=0.7)
ax.plot(theta_opt, range_opt/1e3, '^', color=C['ode45'],  ms=11, zorder=5,
        label=f'R_max = {range_opt/1e3:.0f} km @ θ={theta_opt}°')
ax.plot(45,        range_nd/1e3,  '^', color=C['nodrag'], ms=11, zorder=5,
        label=f'R_max = {range_nd/1e3:.0f} km @ θ=45°')
ax.text(theta_opt+1.5, range_opt/1e3*0.97, f'{theta_opt}°', color=C['ode45'],  fontsize=9)
ax.text(46.5,          range_nd/1e3*0.97,  '45°',          color=C['nodrag'], fontsize=9)
style_ax(ax, title='(c)  Portée maximale  x_max(θ₀)',
         xl='Angle de lancement  θ₀  [°]', yl='Portée  x_max  [km]')
ax.legend(fontsize=8.5, framealpha=0.9)

# ── (d) Apogée y_max(θ) ────────────────────────────────────────────────
ax = fig1.add_subplot(gs1[1, 1])
ax.plot(res_nd[:, 0],   res_nd[:, 2]/1e3,   color=C['nodrag'], lw=2.5, ls='--',
        label='Sans frottement (analytique)', alpha=0.9)
ax.plot(res_drag[:, 0], res_drag[:, 2]/1e3, color=C['ode45'],  lw=2.5,
        label='Avec frottement (ode45)')
# Points expérimentaux
ax.scatter(EXP[:, 0], EXP[:, 2]/1e3, color=C['exp'], s=50, zorder=6,
           marker='s', label='HARP expérimental (Murphy & Bull 1967)')
ax.scatter(EXP[:, 0], sim_apos/1e3,   color=C['hi'],  s=50, zorder=6,
           marker='^', label='Simulation ode45 (mêmes θ, v₀)')
# Annotation Round 28
ax.annotate(f'Round 28\n{APO_R28/1e3:.0f} km (exp)\n{ya_V/1e3:.0f} km (sim)',
            xy=(TH_R28, APO_R28/1e3),
            xytext=(TH_R28 - 20, APO_R28/1e3*0.75),
            fontsize=8, color=C['exp'],
            arrowprops=dict(arrowstyle='->', color=C['exp'], lw=1.3),
            bbox=dict(boxstyle='round,pad=0.3', fc=PANEL, ec=C['exp'], alpha=0.85))
style_ax(ax, title='(d)  Apogée  y_max(θ₀)',
         xl='Angle de lancement  θ₀  [°]', yl='Apogée  y_max  [km]')
ax.legend(fontsize=8.0, framealpha=0.9)

plt.savefig(OUTPUT_DIR / 'harp_fig1_trajectoires.png',
            dpi=130, bbox_inches='tight', facecolor=BG)
print("  ✓  Figure 1 sauvegardée.")


<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">2.7 Temps et position de l’apogée, puis temps et position au point d’impact </h2>
</div>

Pour l’angle optimal obtenu avec la méthode de référence, les résultats principaux sont :

- angle optimal : $50.0^\circ$
- portée maximale : $296.0\ \mathrm{km}$
- apogée : $89.8\ \mathrm{km}$
- position de l’apogée : $(149.2,\ 89.8)\ \mathrm{km}$
- temps d’apogée : $137\ \mathrm{s}$
- temps de vol total : $274\ \mathrm{s}$

Ces quantités on été calculées automatiquement dans les cellules précédentes, puis visualisées dans les figures.


</div>

<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">2.8 Évolution temporelle </h2>
</div>

La figure suivante montre l’évolution temporelle des principales grandeurs :

- $x(t)$ : la position horizontale ;
- $y(t)$ : l’altitude ;
- $v_x(t)$ : la vitesse horizontale ;
- $v_y(t)$ : la vitesse verticale.

On observe que la vitesse horizontale diminue rapidement au début, car la traînée est très forte près du sol. La vitesse verticale décroît pendant la montée, s’annule à l’apogée, puis devient négative pendant la descente.


</div>

In [ ]:

# ══════════════════════════════════════════════════════════════════════
#  8.  FIGURE 2 — Évolution temporelle à θ_opt
# ══════════════════════════════════════════════════════════════════════
fig2 = plt.figure(figsize=(18, 11))
fig2.patch.set_facecolor(BG)
gs2 = gridspec.GridSpec(2, 2, figure=fig2, hspace=0.38, wspace=0.32,
                         left=0.07, right=0.97, top=0.91, bottom=0.08)
fig2.suptitle(
    f"Évolution temporelle  ·  θ_opt = {theta_opt}°  ·  v₀ = {V0_R28:.0f} m/s",
    fontsize=15, fontweight='bold', color=TEXT, y=0.97
)

legend_elements = [
    Line2D([0], [0], color=C['nodrag'], lw=2.5, ls='--', label='Analytique (sans drag)'),
    Line2D([0], [0], color=C['euler'],  lw=2.0,           label='Euler  Δt=0.25 s'),
    Line2D([0], [0], color=C['rk4'],   lw=2.0, ls='-.',  label='RK4    Δt=1.0 s'),
    Line2D([0], [0], color=C['ode45'], lw=2.5,            label='ode45  adaptatif'),
]

# ── (a) x(t) ──────────────────────────────────────────────────────────
ax = fig2.add_subplot(gs2[0, 0])
ax.plot(tA, sA[:, 0]/1e3, color=C['nodrag'], lw=2.5, ls='--', alpha=0.9)
ax.plot(tE, sE[:, 0]/1e3, color=C['euler'],  lw=2.0, alpha=0.9)
ax.plot(tR, sR[:, 0]/1e3, color=C['rk4'],   lw=2.0, ls='-.', alpha=0.9)
ax.plot(tO, sO[:, 0]/1e3, color=C['ode45'], lw=2.5)
# Annotation apogée
ax.axvline(ta1, color='white', ls=':', lw=1, alpha=0.5)
ax.text(ta1+2, sO[np.argmax(sO[:,1]),0]/1e3*0.97, f't_apo={ta1:.0f}s',
        color='white', fontsize=8, va='top')
style_ax(ax, title='(a)  Position horizontale  x(t)',
         xl='Temps  t  [s]', yl='x  [km]')
ax.legend(handles=legend_elements, fontsize=8.5, framealpha=0.9)

# ── (b) y(t) ──────────────────────────────────────────────────────────
ax = fig2.add_subplot(gs2[0, 1])
ax.plot(tA, sA[:, 1]/1e3, color=C['nodrag'], lw=2.5, ls='--', alpha=0.9)
ax.plot(tE, sE[:, 1]/1e3, color=C['euler'],  lw=2.0, alpha=0.9)
ax.plot(tR, sR[:, 1]/1e3, color=C['rk4'],   lw=2.0, ls='-.', alpha=0.9)
ax.plot(tO, sO[:, 1]/1e3, color=C['ode45'], lw=2.5)
# Annotations
ax.plot(ta1, ya1/1e3, '^', color='white', ms=9, zorder=5)
ax.annotate(f'  Apogée\n  t={ta1:.0f}s, y={ya1/1e3:.0f}km',
            xy=(ta1, ya1/1e3), xytext=(ta1-40, ya1/1e3-30),
            fontsize=8, color='white',
            arrowprops=dict(arrowstyle='->', color='white', lw=1.3),
            bbox=dict(boxstyle='round,pad=0.3', fc=PANEL, ec=BORDER, alpha=0.85))
style_ax(ax, title='(b)  Altitude  y(t)',
         xl='Temps  t  [s]', yl='y  [km]')
ax.legend(handles=legend_elements, fontsize=8.5, framealpha=0.9)

# ── (c) vx(t)  [BONUS] ─────────────────────────────────────────────────
ax = fig2.add_subplot(gs2[1, 0])
ax.plot(tA, sA[:, 2], color=C['nodrag'], lw=2.5, ls='--', alpha=0.9, label='Analytique')
ax.plot(tE, sE[:, 2], color=C['euler'],  lw=2.0, alpha=0.9, label='Euler')
ax.plot(tR, sR[:, 2], color=C['rk4'],   lw=2.0, ls='-.', alpha=0.9, label='RK4')
ax.plot(tO, sO[:, 2], color=C['ode45'], lw=2.5, label='ode45')
ax.axhline(0, color=BORDER, lw=1, ls='-')
style_ax(ax, title='(c) [BONUS]  Vitesse horizontale  vₓ(t)',
         xl='Temps  t  [s]', yl='vₓ  [m/s]')
ax.legend(fontsize=8.5, framealpha=0.9)

# ── (d) vy(t)  [BONUS] ─────────────────────────────────────────────────
ax = fig2.add_subplot(gs2[1, 1])
ax.plot(tA, sA[:, 3], color=C['nodrag'], lw=2.5, ls='--', alpha=0.9, label='Analytique')
ax.plot(tE, sE[:, 3], color=C['euler'],  lw=2.0, alpha=0.9, label='Euler')
ax.plot(tR, sR[:, 3], color=C['rk4'],   lw=2.0, ls='-.', alpha=0.9, label='RK4')
ax.plot(tO, sO[:, 3], color=C['ode45'], lw=2.5, label='ode45')
ax.axhline(0, color=BORDER, lw=1, ls='-')
ax.annotate(f'v_y = 0\nApogée',
            xy=(ta1, 0), xytext=(ta1+15, sO[:,3].min()*0.3),
            fontsize=8, color=C['ode45'],
            arrowprops=dict(arrowstyle='->', color=C['ode45'], lw=1.3))
style_ax(ax, title='(d) [BONUS]  Vitesse verticale  v_y(t)',
         xl='Temps  t  [s]', yl='v_y  [m/s]')
ax.legend(fontsize=8.5, framealpha=0.9)

plt.savefig(OUTPUT_DIR / 'harp_fig2_temporel.png',
            dpi=130, bbox_inches='tight', facecolor=BG)
print("  ✓  Figure 2 sauvegardée.")



<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">2.9 BONUS — Courbe, charge et coefficient de traînée </h2>
</div>

Dans la partie bonus, on a inclus :
- une représentation de l’altitude en fonction de la vitesse totale $y(v)$ ;
- l’évolution de la charge structurale en $g$ ;
- un graphique du coefficient de traînée $C_D$ en fonction du Mach.

La charge structurale utilisée dans le code est :
$$
n_g = \frac{|a_{\mathrm{drag}}|}{g_0}
= \frac{\rho(y) C_D A v^2}{2mg_0}.
$$

Le résultat principal est que la charge maximale est d’environ :
$$
n_{g,\max} \approx 10.7\,g,
$$
et qu’elle se produit au lancement, lorsque la vitesse et la densité atmosphérique sont toutes deux maximales.

</div>

In [ ]:

# ══════════════════════════════════════════════════════════════════════
#  9.  FIGURE 3 — BONUS + Validation complète
# ══════════════════════════════════════════════════════════════════════
fig3 = plt.figure(figsize=(18, 13))
fig3.patch.set_facecolor(BG)
gs3 = gridspec.GridSpec(2, 2, figure=fig3, hspace=0.38, wspace=0.34,
                         left=0.07, right=0.97, top=0.91, bottom=0.08)
fig3.suptitle(
    "Analyse avancée  ·  BONUS  ·  Validation expérimentale",
    fontsize=15, fontweight='bold', color=TEXT, y=0.97
)

# ── (a) Validation apogée sim vs exp ──────────────────────────────────
ax = fig3.add_subplot(gs3[0, 0])
lims = [min(EXP[:,2].min(), sim_apos.min())/1e3 * 0.95,
        max(EXP[:,2].max(), sim_apos.max())/1e3 * 1.05]
ax.plot(lims, lims, color=MUTED, lw=1.5, ls='--', label='Simulation = Expérience')
ax.scatter(EXP[:, 2]/1e3, sim_apos/1e3, color=C['ode45'], s=80, zorder=5,
           label='Tirs HARP (ode45, C_D constant)', edgecolors='white', lw=0.7)
# Round 28 en surbrillance
ax.scatter([APO_R28/1e3], [ya_V/1e3], color=C['exp'], s=150, zorder=6,
           marker='*', label=f'Round 28  (Δ={abs(ya_V-APO_R28)/APO_R28*100:.1f}%)')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_aspect('equal')
style_ax(ax, title='(a)  Validation — Apogée  sim. vs. exp.',
         xl='Apogée expérimental  [km]', yl='Apogée simulé  [km]')
ax.legend(fontsize=8.5, framealpha=0.9)
# Erreur moyenne
ax.text(0.05, 0.95, f'Erreur moy. = {np.mean(np.abs(errs_apo)):.1f}%',
        transform=ax.transAxes, fontsize=9, color=C['ode45'], va='top',
        bbox=dict(boxstyle='round', fc=PANEL, ec=C['ode45'], alpha=0.8))

# ── (b) Altitude vs vitesse totale — y(v)  [BONUS] ─────────────────────
ax = fig3.add_subplot(gs3[0, 1])
# Monter = aller, descendre = retour  (couleur différente)
i_apo = np.argmax(sO[:, 1])
ax.plot(vO[:i_apo]/1e3, sO[:i_apo, 1]/1e3, color=C['ode45'], lw=2.5,
        label='Montée (lancement → apogée)')
ax.plot(vO[i_apo:]/1e3, sO[i_apo:, 1]/1e3, color=C['hi'], lw=2.5, ls='--',
        label='Descente (apogée → impact)')
# Marqueurs clés
ax.plot(vO[0]/1e3,    sO[0, 1]/1e3,    'o', color='white', ms=8, zorder=5,
        label=f'Lancement  v={vO[0]/1e3:.2f} km/s')
ax.plot(vO[i_apo]/1e3, sO[i_apo, 1]/1e3, '^', color='white', ms=8, zorder=5,
        label=f'Apogée  v={vO[i_apo]/1e3:.3f} km/s')
ax.plot(vO[-1]/1e3,   sO[-1, 1]/1e3,   '*', color=C['exp'], ms=10, zorder=5,
        label=f'Impact  v={vO[-1]/1e3:.2f} km/s')
# Vitesse du son (Mach 1 ≈ 340 m/s)
ax.axvline(sound_speed(0)/1e3, color=MUTED, ls=':', lw=1.5, alpha=0.7)
ax.text(sound_speed(0)/1e3 + 0.01, sO[:, 1].max()*0.5/1e3, 'Mach 1',
        color=MUTED, fontsize=8, rotation=90, va='center')
style_ax(ax, title=f'(b) [BONUS]  Altitude vs vitesse  y(v)  —  θ={theta_opt}°',
         xl='Vitesse totale  |v|  [km/s]', yl='Altitude  y  [km]')
ax.legend(fontsize=8, framealpha=0.9)

# ── (c) Charge g  [BONUS] ──────────────────────────────────────────────
ax = fig3.add_subplot(gs3[1, 0])
ax.fill_between(tO, gl_opt, color=C['gload'], alpha=0.25)
ax.plot(tO, gl_opt, color=C['gload'], lw=2.5, label=f'Charge g — θ={theta_opt}°')

# Valider aussi pour Round 28 (quasi-vertical)
gl_r28 = compute_gload(tV, sV)
ax.plot(tV, gl_r28, color=C['exp'], lw=2.0, ls='--', alpha=0.8,
        label=f'Charge g — θ={TH_R28}° (Round 28)')

ax.axhline(1, color=MUTED, ls=':', lw=1, alpha=0.7)
ax.text(tO[-1]*0.02, gl_opt.max()*0.7,
        f'G_max = {gl_opt.max():.1f} g\n(à t={tO[np.argmax(gl_opt)]:.0f} s)',
        fontsize=9, color=C['gload'],
        bbox=dict(boxstyle='round', fc=PANEL, ec=C['gload'], alpha=0.85))
style_ax(ax, title='(c) [BONUS]  Charge g structurale  n_g(t)',
         xl='Temps  t  [s]', yl='Charge g  [× g₀]')
ax.legend(fontsize=8.5, framealpha=0.9)

# ── (d) CD(Mach) vs formule  [BONUS] ───────────────────────────────────
ax = fig3.add_subplot(gs3[1, 1])
M_range = np.linspace(0.3, 8, 400)
cd_formula = np.array([cd_mach(m * 340, 0) for m in M_range])
ax.plot(M_range, cd_formula, color=C['nodrag'], lw=2.5,
        label='C_D(M) — formule notes de cours (slide II-35)')
ax.axhline(CD_CONST, color=C['ode45'], lw=2.5, ls='--',
           label=f'C_D = {CD_CONST:.4f}  (Murphy & Bull 1967, Round 28)')
# Mach au lancement de Round 28
M_launch = V0_R28 / sound_speed(0)
ax.axvline(M_launch, color=C['exp'], ls=':', lw=1.5, alpha=0.8)
ax.text(M_launch + 0.1, 0.45, f'Mach lancement\nRound 28\nM={M_launch:.1f}',
        fontsize=8, color=C['exp'], va='top',
        bbox=dict(boxstyle='round', fc=PANEL, ec=C['exp'], alpha=0.8))
# Régimes
for x_lo, x_hi, label, col in [
    (0.3, 0.85, 'Sub.', '#2a4a6a'),
    (0.85, 1.2, 'Trans.', '#4a2a2a'),
    (1.2, 5.0, 'Supersonique', '#1a3a1a'),
    (5.0, 8.0, 'Hyper.', '#3a1a3a'),
]:
    ax.axvspan(x_lo, x_hi, alpha=0.10, color=col)
    ax.text((x_lo+x_hi)/2, 0.47, label, ha='center', fontsize=7, color=MUTED)
style_ax(ax, title='(d) [BONUS]  Coefficient de traînée  C_D(M)',
         xl='Nombre de Mach  M  [—]', yl='C_D  [—]')
ax.legend(fontsize=8.5, framealpha=0.9)
ax.set_ylim(0, 0.55)
ax.set_xlim(0.3, 8)

plt.savefig(OUTPUT_DIR / 'harp_fig3_bonus_validation.png',
            dpi=130, bbox_inches='tight', facecolor=BG)
print("  ✓  Figure 3 sauvegardée.")

<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">2.10 Tableau récapitulatif des résultats numériques </h2>
  
</div>


La cellule suivante imprime un résumé final de toutes les grandeurs importantes obtenues dans la simulation :
- angle optimal
- portée
- apogée
- temps d’apogée
- temps de vol
- validation Round 28
- résultats bonus

</div>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  10.  TABLEAU RÉCAPITULATIF
# ══════════════════════════════════════════════════════════════════════
print("\n" + "═" * 64)
print("  TABLEAU DE RÉSULTATS — DEVOIR #3")
print("═" * 64)
print(f"\n  Projectile   : HARP Martlet 2C")
print(f"  Masse        : {M_PROJ} kg            Diamètre : {D_PROJ*100:.1f} cm")
print(f"  C_D          : {CD_CONST:.4f} (const)  A = {A_PROJ*1e4:.2f} cm²")
print(f"  v₀ (Round 28): {V0_R28:.0f} m/s  (Mach {V0_R28/sound_speed(0):.1f})")
print(f"\n  Angle optimal (portée max, ode45) :  θ_opt = {theta_opt}°")
print(f"  Sans frottement (analytique)      :  θ_opt = {theta_nd}°  (attendu 45°)")
print(f"\n  À θ_opt = {theta_opt}°  (ode45, v₀={V0_R28:.0f} m/s) :")
print(f"    Portée max         : {range_opt/1e3:.1f} km")
print(f"    Apogée             : {apo_opt/1e3:.1f} km")
_, xa_o, ya_o, tl_o, xr_o = get_key_points(tO, sO)
print(f"    Pos. apogée        : ({xa_o/1e3:.1f}, {ya_o/1e3:.1f}) km")
print(f"    Temps d'apogée     : {ta1:.0f} s")
print(f"    Temps de vol total : {tl_o:.0f} s")
print(f"\n  Validation Round 28 (θ={TH_R28}°, v₀={V0_R28:.0f} m/s) :")
print(f"    Apogée sim.  : {ya_V/1e3:.1f} km  |  exp.  : {APO_R28/1e3:.1f} km  |  Δ = {abs(ya_V-APO_R28)/APO_R28*100:.1f}%")
print(f"    Portée sim.  : {xr_V/1e3:.1f} km  |  exp.  : {RNG_R28/1e3:.1f} km  |  Δ = {abs(xr_V-RNG_R28)/RNG_R28*100:.1f}%")
print(f"\n  BONUS :")
print(f"    G-load max en vol  : {gl_opt.max():.1f} g  (à t={tO[np.argmax(gl_opt)]:.0f} s)")
print(f"    Mach au lancement  : {MachO[0]:.2f}")
print(f"    Vitesse d'impact   : {vO[-1]:.0f} m/s  (Mach {MachO[-1]:.2f})")
print(f"\n  Figures sauvegardées dans {OUTPUT_DIR}")
print("═" * 64)

plt.show()


<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">3. Validation expérimentale </h2>
  
</div>

On peut faire une comparaison de notre simulation théorique avec des données expérimentales HARP.

Le cas principal de validation est le Round 28, avec :
$$
\theta = 84.8^\circ,\qquad v_0 = 2164\ \mathrm{m/s}.
$$

Les résultats obtenus dans le code sont :

- apogée simulée : $169.3\ \mathrm{km}$
- apogée expérimentale : $179.8\ \mathrm{km}$
- erreur sur l’apogée : $5.8\%\$

- portée simulée : $61.6\ \mathrm{km}$
- portée expérimentale : $57.3\ \mathrm{km}$
- erreur sur la portée : $7.5\%$

La validation sur l’ensemble des tirs HARP donne aussi des erreurs moyennes de quelques pourcents, ce qui montre que le modèle, bien que simplifié, capture correctement les tendances globales.

</div>

<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">4. Analyse et discussion des résultats </h2>
  
</div>

Les résultats obtenus montrent d’abord que la traînée atmosphérique joue un rôle majeur dans la trajectoire du projectile. Le modèle sans frottement surestime fortement la portée et l’apogée, ce qui confirme qu’un traitement purement parabolique serait inadéquat pour un projectile lancé à plus de $2\ \mathrm{km/s}$.

Ensuite, l’angle optimal de lancement n’est plus $45^\circ$, mais environ $50^\circ$. Ce décalage est physiquement cohérent : une trajectoire légèrement plus verticale permet au projectile de sortir plus rapidement des couches basses de l’atmosphère, là où les pertes aérodynamiques sont les plus fortes.

La comparaison entre Euler, RK4 et RK45 montre aussi que les méthodes de plus haut ordre donnent pratiquement les mêmes résultats. Cela renforce la confiance dans la stabilité numérique de la simulation. Euler reste utile comme comparaison, mais il est un peu moins précis.

Finalement, la validation expérimentale est satisfaisante. Les écarts de quelques pourcents sont plausibles compte tenu des simplifications du modèle :
- mouvement strictement 2D ;
- coefficient de traînée constant dans le cas principal ;
- absence de vent
- absence de portance
- atmosphère exponentielle simplifiée

</div>

<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">5. Conclusion </h2>
  
</div>

Dans ce travail, on a étudié la trajectoire balistique du projectile HARP Martlet 2C à l’aide d’un modèle numérique comprenant :
- une gravité variable avec l’altitude
- une atmosphère exponentielle
- une traînée quadratique de Newton

Le résultat principal est que l’angle de lancement donnant la portée maximale vaut environ :
$$
\theta_{\mathrm{opt}} \approx 50^\circ.
$$

Pour cet angle, les résultats numériques obtenus sont :
- portée maximale : $296.0\ \mathrm{km}$
- apogée : $89.8\ \mathrm{km}$
- temps d’apogée : $137\ \mathrm{s}$
- temps de vol total : $274\ \mathrm{s}$

Les analyses bonus montrent aussi que la charge maximale subie par le projectile atteint environ $10.7\,g\$ au lancement. Enfin, la comparaison avec les données expérimentales HARP montre que le modèle reste réaliste, avec des erreurs de quelques pourcents seulement. Ce travail met en évidence l’importance de la traînée atmosphérique dans l’analyse balistique d’un projectile réel lancé à très haute vitesse.

</div>

<div style="margin: 40px 0;">
    <div style="height: 3px; background: linear-gradient(90deg, transparent, #2196F3, #9C27B0, #2196F3, transparent); border-radius: 3px;"></div>
</div>

<a id="intro"></a>

<div style="
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 100%);
    padding: 22px 30px;
    border-radius: 12px;
    margin: 30px 0 20px 0;
    box-shadow: 0 8px 25px rgba(15, 12, 41, 0.5);
    border: 1px solid rgba(168, 208, 230, 0.15);
">
    <h2 style="color: white; margin: 0; text-align: center; font-size: 24px; font-weight: 300; letter-spacing: 2px; text-shadow: 0 2px 8px rgba(0,0,0,0.3);">6. Sources </h2>
  
</div>

Les principales sources utilisées dans ce travail sont :

1. **Énoncé du devoir 3** du cours PHY-2100 — Sciences de l’espace  
2. **Notes de cours TIRS** et le code de base mentionné dans l’énoncé  
3. **Murphy & Bull (1967)** pour les données HARP et la validation expérimentale  